### Generative AI- Customer Churn Reports

This notebook implements two types of templates using Gimini API to generate customer churn risk summary and retention recommendation, The generated reports will be tested and evaluated to identify which prompt template produces more suitable output.

**Prompt Templates:

-- T1- Zero-Shot: Generates repots using instructions without examples

-- T2- Few-Shot: Generates repots using instructions with worked examples

Both templates use the same instructions ans customet information to support a fair comparision

In [74]:
from pathlib import Path

template_path=Path("Generative_AI/prompts/zero_shot_template.txt")

zero_shot_template = template_path.read_text( encoding="utf-8")

print("Zero-shot template loaded seccessfully")

Zero-shot template loaded seccessfully


In [75]:
from pathlib import Path

template_path=Path("Generative_AI/prompts/few_shot_template.txt")

few_shot_template = template_path.read_text( encoding="utf-8")

print("Few-shot template loaded successfully")

Few-shot template loaded successfully


In [76]:
# prepare sample customer data for testing
customer = {
    "churn_prediction": 1,
    "churn_probability": "87%",
    "recency_days":45,
    "loyalty_score": 25,
    "cart_abandonment_rate": "70%"
}
# fill both prompts with the sample customer data
filled_zero_shot = zero_shot_template.format(**customer)
filled_few_shot = few_shot_template.format(**customer)

In [77]:
# check the filled zero-shot prompt 
print(filled_zero_shot)

You are a customer retention analyst preparing a concise report for business executives.

Task:
Analyze the customer's churn prediction and available customer features. Provide practical retention recommendations


Customer Information:
- Churn Prediction: 1
- Churn Probability: 87%
- Recency Days: 45
- Loyalty Score: 25
- Cart Abandonment Rate: 70%


Instructions:
1. Summarize the customer's estimated churn risk.
2. Identify relevant observations from the provided customer information.
3. Recommend exactly two actionable retention strategies tailored to the customer.
4. Briefly explain the rationale for each strategy.
5. Use clear, professional, executive-friendly English.


Constraints:
- Use only the information provided.
- Do not invent customer details or business facts.
- Do not assume that observed features cause churn.
- Treat churn probability as a model estimate, not a certainty.
- Avoid unsupported guarantees about outcomes.
- If information is insufficient, state the limita

In [84]:
# check the filled few-shot prompt 
print(filled_few_shot)


You are a customer retention analyst preparing a concise report for business executives.

Task:
Analyze the customer's churn prediction and available customer features. Provide practical retention recommendations.

Instructions:
1. Summarize the customer's estimated churn risk.
2. Identify relevant observations from the provided customer information.
3. Recommend exactly two actionable retention strategies tailored to the customer.
4. Briefly explain the rationale for each strategy.
5. Use clear, professional, executive-friendly English.

Constraints:
- Use only the provided customer information.
- Do not invent facts about the customer.
- Do not claim that customer features cause churn.
- Treat churn probability as a model estimate, not a guaranteed outcome.
- If information is insufficient, acknowledge it.
- Do not include additional sections.

Required Output Format:

Risk Summary:
[Summarize the predicted churn risk]

Retention Recommendations:
1. [Action and brief rationale]
2. [

In [88]:
from Generative_AI.api_config_template import client, MODEL_NAME

#Prepare function to generate content using the Gemini API
def generate_customer_reports(customer_data):

    # Fill the prompts with the customer data
    filled_zero_shot = zero_shot_template.format(**customer_data)
    filled_few_shot = few_shot_template.format(**customer_data)

    # Generate zero-shot report
    zero_shot_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=filled_zero_shot
    )
    zero_shot_report = zero_shot_response.text

    # Generate few-shot report
    few_shot_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=filled_few_shot
    )
    few_shot_report = few_shot_response.text

    # Return both reports
    return {
        "zero_shot_report": zero_shot_report,
        "few_shot_report": few_shot_report
    }

Gemini API key loaded successfully!


In [89]:
# Example: Generate both reports for a mock customer
reports = generate_customer_reports(customer)

print("ZERO-SHOT REPORT:")
print(reports["zero_shot_report"])

print("\nFEW-SHOT REPORT:")
print(reports["few_shot_report"])

ZERO-SHOT REPORT:
Risk Summary:
The predictive model estimates an 87% churn probability, classifying the customer as a high-risk profile for discontinuing engagement.

Retention Recommendations:
1. Deploy targeted checkout assistance or personalized cart recovery incentives, justified by the customer's 70% cart abandonment rate.
2. Initiate proactive re-engagement outreach within the next few days, addressing the 45-day recency gap and the low loyalty score of 25.

Strategic Rationale:
The recommendations directly address the customer's observed behavioral indicators—specifically the high cart abandonment rate, moderate-to-high recency window, and low loyalty score—aiming to re-stimulate purchasing activity and improve engagement before churn occurs.

FEW-SHOT REPORT:
Risk Summary:
The model estimates an 87% churn probability for this customer, indicating elevated predicted churn risk. The available data show 45 recency days, a loyalty score of 25, and a cart abandonment rate of 70%.

